## 27. Out-of-Time Temporal Backtest

The validation above uses a random train/test split, which doesn't test whether the model holds up *forward in time* — the more realistic way a PD model actually gets used in production. The first attempt at this used a forward-looking split (train 2015–18, validate 2019, test 2020–22) that assumed the dataset extended past 2018. It doesn't — the source file is literally named `accepted_2007_to_2018Q4.csv.gz`, and the 2019+ years simply don't exist in it. The corrected split below shifts the same train/validate/test structure two years earlier, onto years that actually exist in the data.

In [ ]:
# ============================================================
# CELL 6 (corrected) — Rebuild the temporal split using years
# that actually exist in this dataset (2007-2018Q4 only)
# ============================================================

# The original roadmap plan (train 2015-18, validate 2019, test 2020-22) is
# impossible with this dataset -- the file is literally named
# accepted_2007_to_2018Q4.csv.gz and contains zero loans originated after
# Q4 2018. Shifting the windows two years earlier preserves the same
# train/validate/test structure and relative sizing, using only years that
# actually exist in the data.

issue_dates_temporal = pd.to_datetime(df_cleaned['issue_d'], format='mixed')
df_cleaned['issue_year_temporal'] = issue_dates_temporal.dt.year

train_mask = df_cleaned['issue_year_temporal'].between(2013, 2016)
val_mask   = df_cleaned['issue_year_temporal'] == 2017
test_mask  = df_cleaned['issue_year_temporal'] == 2018

print("--- Temporal split sizes (corrected to available years) ---")
print(f"Train (2013-2016): {train_mask.sum():,} loans")
print(f"Validate (2017):   {val_mask.sum():,} loans")
print(f"Test (2018):       {test_mask.sum():,} loans")

print("\n--- Default rate by period (sanity check before modeling) ---")
for label, mask in [('Train', train_mask), ('Validate', val_mask), ('Test', test_mask)]:
    rate = df_cleaned.loc[mask, 'default'].mean()
    print(f"{label}: {rate:.4f}")

In [ ]:
# ============================================================
# CELL 7 (fixed) — Train a fresh LightGBM model on ONLY 2013-2016 data
# ============================================================
model_df_temporal = df_cleaned[feature_cols + ['default']].dropna()

# Rebuild the year masks directly on model_df_temporal's index, since dropna()
# above removed some rows and the original train_mask/val_mask/test_mask
# (built on the full df_cleaned) no longer align in length.
issue_year_aligned = df_cleaned['issue_year_temporal'].reindex(model_df_temporal.index)

train_mask_aligned = issue_year_aligned.between(2013, 2016)
val_mask_aligned   = issue_year_aligned == 2017
test_mask_aligned  = issue_year_aligned == 2018

X_train_temporal = model_df_temporal.loc[train_mask_aligned, feature_cols]
y_train_temporal = model_df_temporal.loc[train_mask_aligned, 'default']

X_val_temporal = model_df_temporal.loc[val_mask_aligned, feature_cols]
y_val_temporal = model_df_temporal.loc[val_mask_aligned, 'default']

X_test_temporal = model_df_temporal.loc[test_mask_aligned, feature_cols]
y_test_temporal = model_df_temporal.loc[test_mask_aligned, 'default']

print(f"Train: {len(X_train_temporal):,} | Validate: {len(X_val_temporal):,} | Test: {len(X_test_temporal):,}")

lgb_temporal = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6, num_leaves=31,
    min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=len(y_train_temporal[y_train_temporal==0]) / len(y_train_temporal[y_train_temporal==1]),
    random_state=42
)
lgb_temporal.fit(X_train_temporal, y_train_temporal)

calibrated_lgb_temporal = CalibratedClassifierCV(FrozenEstimator(lgb_temporal), method='sigmoid', cv='prefit')
calibrated_lgb_temporal.fit(X_val_temporal, y_val_temporal)

print("Temporal model trained and calibrated.")

In [ ]:
# ============================================================
# CELL 8 — Evaluate: does AUC degrade from validation (2017) to test (2018)?
# ============================================================
test_probs_temporal = calibrated_lgb_temporal.predict_proba(X_test_temporal)[:, 1]
auc_temporal_test = roc_auc_score(y_test_temporal, test_probs_temporal)

val_probs_temporal = calibrated_lgb_temporal.predict_proba(X_val_temporal)[:, 1]
auc_temporal_val = roc_auc_score(y_val_temporal, val_probs_temporal)

print("--- Temporal Backtest: AUC Comparison ---")
print(f"AUC on 2017 (validation):  {auc_temporal_val:.4f}")
print(f"AUC on 2018 (test):        {auc_temporal_test:.4f}")
print(f"AUC change: {auc_temporal_test - auc_temporal_val:+.4f}")
print(f"\nNote: 2018 test-set default rate (15.76%) is lower than both train (20.09%)")
print(f"and validation (23.13%) -- likely reflects seasoning bias (slower-resolving")
print(f"defaults haven't appeared yet in this snapshot), not necessarily a genuine")
print(f"improvement in 2018 loan quality. Interpret AUC change with this in mind.")

fpr_t, tpr_t, _ = roc_curve(y_test_temporal, test_probs_temporal)
fpr_v, tpr_v, _ = roc_curve(y_val_temporal, val_probs_temporal)
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr_t, tpr_t, color='#e74c3c', linewidth=2, label=f'2018 test (AUC={auc_temporal_test:.3f})')
ax.plot(fpr_v, tpr_v, color='#3498db', linewidth=2, label=f'2017 validation (AUC={auc_temporal_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Temporal Backtest: 2017 Validation vs 2018 Test')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9 — Real PSI check: 2013-2016 training population vs 2018 test population
# ============================================================
train_probs_temporal = calibrated_lgb_temporal.predict_proba(X_train_temporal)[:, 1]
psi_temporal = calculate_psi(train_probs_temporal, test_probs_temporal)

print("--- Population Stability Index: 2013-2016 training scores vs 2018 test scores ---")
print(f"PSI: {psi_temporal:.4f}")
print(f"Interpretation: PSI < 0.1 = stable, 0.1-0.2 = moderate shift, > 0.2 = unstable.")

## Summary

| Model | Metric | Value |
|---|---|---|
| Logistic Regression Scorecard | AUC | 0.701 |
| LightGBM | AUC | 0.713 |
| LightGBM | Gini | 0.425 |
| LightGBM | KS | 0.309 |
| Two-Stage LGD (v2) | RMSE | 0.019 |

**Portfolio Expected Loss:** $1.34B on $19.4B exposure (6.89% EL rate). Grade G EL rate (24.65%) is ~22x Grade A (1.09%), driven by the multiplicative effect of higher PD, LGD, and EAD all moving together.

**Regulatory capital:** IFRS 9 provisions and Basel III IRB capital come out roughly comparable in magnitude ($1.45B vs $1.27B) — expected and unexpected loss cushions of similar size for this portfolio.

**Portfolio simulation:** the Gaussian copula Monte Carlo's 99.9% VaR exceeds the point-estimate Expected Loss by more than 2.5x, which is the entire reason regulators require a tail-risk capital buffer on top of expected-loss provisions in the first place.

**Out-of-time backtest** (train 2013–16, validate 2017, test 2018): AUC change of only +0.0013 and PSI of 0.025 — no meaningful degradation or population drift across the time split.

### Known limitations

This project deliberately documents (rather than hides) several real modeling limitations encountered along the way:

- **Macro sensitivity could not be reliably estimated from this dataset.** A time-varying Cox model's fitted unemployment hazard ratio was directionally implausible because origination vintage and macro conditions are too collinear in this sample to separate. Stress testing uses a literature-based hazard ratio instead of the dataset's own unreliable estimate.
- **The dataset doesn't extend past Q4 2018**, which constrains how far forward an out-of-time backtest can actually reach.
- **The copula correlation parameter (rho) calibration was paused, not completed** — a likely seasoning-bias contaminant in the observed default-rate variance was flagged but not yet quantified, so the Basel-prescribed value (rho = 0.12) is used as a working anchor.
- **EAD for live loans in the IFRS 9 section uses original `loan_amnt`** rather than current outstanding balance (`out_prncp` wasn't available in that data pull), which overstates exposure for partially paid-down loans — the provisions total there is an upper bound, not a precise figure.
- **The Distance-to-Default time series** couldn't be extended back to the COVID-19 period due to SEC EDGAR XBRL tag inconsistencies across older filings; the ~24-month window achieved still captures a real 2025 market-stress episode and validates the core premise.

### Possible extensions

- LLM-based feature extraction from the free-text `desc` column (financial stress score, tone, red flags)
- Resolving the deferred copula correlation calibration once the seasoning-bias contamination is quantified
- Pulling `out_prncp` into the live-loan IFRS 9 pipeline for a more precise EAD
- An LLM-as-credit-analyst benchmark comparing model PD against an LLM's qualitative read of borrower descriptions
